# Double Q Network

Q-values estimated by an agent can often be noisy, leading to the overestimation of certain actions. When using an artificial neural network (ANN) to approximate Q-values, this overestimation can occur unevenly across different actions. As a result, the agent may prefer suboptimal actions that appear falsely valuable, which slows down learning and may lead to a poor policy.

To mitigate this, we use Double Q-learning, which involves maintaining two separate Q-value estimators. The key idea is that the likelihood of both estimators simultaneously overestimating the same action is lower. In our implementation, we create two identical neural networks, each representing a Q-table. These networks are trained on different mini-batches of experience to keep their updates distinct.

Here’s how it works during training:

- Network Q is used to estimate Q-values for all actions in the current state 
𝑆
S.

- Network R is used to evaluate the Q-values of the next state 
𝑆
′
S 
′
 , which helps in computing the target Q-value for 
𝑆
S using the Bellman equation.

This separation helps reduce overestimation bias, leading to more stable learning and better policy convergence over time.

In [ ]:
def init_net(df,lkbk,START_IDX,max_mem,reward_function):
    """
    This initialises the RL run by instantiating a new Game,
    creating a new predictive neural network and instantiating
    experience replay.
    Args:
        df: This is the data frame with the market data
        lkbk: This is the lookback period, eg. a value of 10 means 10mins, 10hrs and 10days!
        START_IDX: This is the starting index for the main loop, allow enough for lkbk

    Returns:
        env: an instance of Game, our environment
        model: the neural network
        exp_replay: an instance of ExperienceReplay
    """

    bars1h = df['close'].resample('1h',label='right',closed='right').ohlc().dropna()
    bars1d = df['close'].resample('1D',label='right',closed='right').ohlc().dropna()
    env = Game(df, bars1d,bars1h,reward_function, lkbk=lkbk, init_idx=START_IDX)
    hidden_size = len(env.state)*HIDDEN_MULT


    # Double Q-Network
    
    # Neural Network 1
    model = Sequential()
    model.add(Input(shape=(len(env.state),)))
    model.add(Dense(len(env.state), activation=ACTIVATION_FUN))
    model.add(Dense(hidden_size, activation=ACTIVATION_FUN))
    model.add(Dense(NUM_ACTIONS, activation='softmax'))
    model.compile(loss=LOSS_FUNCTION)

    # Neural Network 2
    model2 = Sequential()
    model2.add(Input(shape=(len(env.state),)))
    model2.add(Dense(len(env.state), activation=ACTIVATION_FUN))
    model2.add(Dense(hidden_size, activation=ACTIVATION_FUN))
    model2.add(Dense(NUM_ACTIONS, activation='softmax'))
    model2.compile(loss=LOSS_FUNCTION)

    exp_replay = ExperienceReplay(max_memory=max_mem)

    return env,model,model2,exp_replay